In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import os
import torch
import pickle
import config

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from torch.utils.data import  DataLoader

import torch.nn.functional as F


from src.metric import *
from src.bi_encoder_training import cosent_loss,compute_batch_embeddings
from src.datasets import ResumeJDDataset

In [3]:
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")
torch.set_float32_matmul_precision("high")

In [4]:
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [5]:
path=config.CLEANED_DATA_DIR

In [6]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    
    

In [7]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=config.device)
optimizer = torch.optim.AdamW(bi_encoder.parameters(), lr=2e-5)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [9]:
g = torch.Generator()
g.manual_seed(config.SEED)

In [10]:
labels=[float(config.label_to_score[label]) for label in train_df['label']]
train_dataset=ResumeJDDataset(train_df['resume_text'].values,train_df['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [11]:
val_labels=[float(config.label_to_score[label]) for label in val_df['label']]
val_dataset=ResumeJDDataset(val_df['resume_text'].values,val_df['job_description_text'].values,val_labels)

val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [12]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=5
patience=2

best_model_path=os.path.join(config.CHUNKED_MODEL_DIR,'bi_encoder_chunked')
os.makedirs(config.CHUNKED_MODEL_DIR,exist_ok=True)

In [13]:
resume_chunk_map,jd_chunk_map={},{}

In [14]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    bi_encoder.train()
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        with autocast(device_type="cuda", dtype=torch.float16):
            resume_embs, jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
            
            labels=labels.to(config.device)
            loss=cosent_loss(resume_embs,jd_embs,labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Training Loss:",average_loss)
        
        
    #Validation    
    bi_encoder.eval()
    val_loss=0
    scores=[]
    
    
    with torch.no_grad():
        
        val_progress_bar = tqdm(val_loader, desc="Validation")
        for resumes,jds,labels in val_progress_bar:
            
            with autocast(device_type="cuda", dtype=torch.float16):    
                val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds,resume_chunk_map,jd_chunk_map)
                
                labels=labels.to(config.device)
                loss=cosent_loss(val_resume_embs,val_jd_embs,labels)
            
                val_scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
                val_scores=val_scores.cpu().numpy()
            
            val_loss += loss.item()
            scores.extend(val_scores)
            val_progress_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        print("Average Validation Loss:",val_loss/len(val_progress_bar))
        
        metrics=model_evaluation(scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        bi_encoder.save(best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

Epoch 1/5


Training:   0%|          | 0/780 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


Average Training Loss: 2.590694899436755


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

...

Epoch 5/5


Training:   0%|          | 0/780 [00:00<?, ?it/s]

Average Training Loss: 1.6179746647437032


Validation:   0%|          | 0/146 [00:00<?, ?it/s]

Average Validation Loss: 0.02509998132104743
NDCG: 0.8557302102330234
MAP: 0.9552756190181384


In [15]:
bi_encoder=SentenceTransformer(best_model_path,device=config.device)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
bi_encoder.eval()

with torch.no_grad():
    with autocast(device_type="cuda", dtype=torch.float16):
        val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,val_df['resume_text'].values,
                                                          val_df['job_description_text'].values,resume_chunk_map,jd_chunk_map)
    
        scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
        scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,val_df,"job_description_text")

In [17]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.7300007063999402
Top-3 Accuracy: 1.0
NDCG: 0.8571881197556811
MAP: 0.9534553058001247
MRR: 0.96875


In [18]:
eval_df=val_df.copy()
eval_df['score']=scores
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 0.620
% groups with spread < 0.1:0.000


In [19]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 0

Sample false_neg resumes:


In [20]:
confused = eval_df[(eval_df['label'] == 1) & (eval_df['score'] < 1.2) & (eval_df['score'] > -0.1)]

print(f"confused predictions: {len(confused)}")
print("\nSample confused resumes:")

i=0
for jd,group in confused.groupby('job_description_text'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

confused predictions: 285

Sample confused resumes:
----------------------------------------------------------------------------------------------------
JD:  Experienced in Salesforce Industries Communications cloud.Certification a plus*10+ years in SFDC 5+ years of experience in Telecom domain solutioning for Quoteto Cash Architect software solutions usi
Score: 0.732
Resume: Professional SummaryBusiness Intelligence Consultant with a 10-year career in data warehousing, business intelligence reporting, and data management architecture. Progressive developer and technical team lead with a strength in design & development, as well as driving performance, reducing inefficie

Score: 0.736
Resume: SummaryExperienced Data Analyst who responds to shifting business needs and priorities in a systematic and effective way.  Excels at implementing operational assessments and conducting functional requirements analysis for businesses of all sized.  Committed to maintaining cutting edge technical sk

In [21]:
resume_chunk_map,jd_chunk_map={},{}

In [22]:

with torch.no_grad():
        
    test_resume_embs, test_jd_embs=compute_batch_embeddings(bi_encoder,test_df['resume_text'].values,
                                                            test_df['job_description_text'].values,resume_chunk_map,jd_chunk_map)
    
    scores=F.cosine_similarity(test_resume_embs,test_jd_embs,dim=1)
    scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [23]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.5289604163874718
Top-3 Accuracy: 0.9642857142857143
NDCG: 0.7012435103836991
MAP: 0.8304389446398798
MRR: 0.8566815697963239
